In [ ]:
repository_filter: list[str] = []
total_repositories_scanned: int = 0
data_file_node: str = "../samples/v2/org.openrewrite.node.dependencies.table.EndOfLifeDependencyReport.csv"
data_file_csharp: str = "../samples/v2/org.openrewrite.csharp.dependencies.table.EndOfLifeDependencyReport.csv"

In [ ]:
from collections.abc import Callable

from moderne_visualizations_misc.reusable.data_loader import (
    read_data_table,
    read_optional_csv,
)
from moderne_visualizations_misc.reusable import quality_utils as qu
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.simplefilter("ignore")

REPORT_COLUMNS = [
    "product",
    "release",
    "eolFrom",
    "eol",
    "daysUntilEol",
    "direct",
    "dependencyPath",
]


def group_artifact_id(df: pd.DataFrame) -> pd.Series:
    """Maven coordinates as `groupId:artifactId`."""
    artifact = df["artifactId"].fillna("").astype(str)
    group = df["groupId"].fillna("").astype(str).str.strip()
    return group.where(group == "", group + ":") + artifact


def normalize(
    df: pd.DataFrame,
    technology: str,
    component_id: Callable[[pd.DataFrame], pd.Series],
) -> pd.DataFrame:
    """Map one language-specific EOL report onto the shared component schema."""
    if df.empty:
        return pd.DataFrame()
    df = qu.add_repo_short(qu.filter_repos(df, repository_filter))
    if df.empty:
        return pd.DataFrame()
    df = df.copy()
    df["technology"] = technology
    df["componentId"] = component_id(df)
    keep = ["repoShort", "technology", "componentId", "version"]
    return df[keep + [c for c in REPORT_COLUMNS if c in df.columns]]


def read_primary(sample: str) -> pd.DataFrame:
    """Read the primary data table, tolerating a missing or empty file.

    A run on a portfolio without Java produces no Java EOL table, and this
    dashboard must still render from the other language-specific reports.
    """
    try:
        return read_data_table(sample)
    except (FileNotFoundError, pd.errors.EmptyDataError):
        return pd.DataFrame()


# Primary data table (injected via NB_DATA_TABLE env var when run by the platform)
# plus the language-specific reports (injected via papermill parameters).
frames = [
    normalize(
        read_primary(
            "../samples/v2/org.openrewrite.java.dependencies.endoflife.table.EndOfLifeDependencyReport.csv"
        ),
        "Java",
        group_artifact_id,
    ),
    normalize(read_optional_csv(data_file_node), "Node.js", lambda d: d["packageName"]),
    normalize(read_optional_csv(data_file_csharp), ".NET", lambda d: d["packageId"]),
]
frames = [f for f in frames if not f.empty]
eol_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

In [ ]:
RED = "#EF5350"  # already end-of-life
AMBER = "#FFB74D"  # approaching end-of-life
# Fixed technology hues (identity, not rank) so filters never repaint a slice.
TECH_COLORS = {"Java": "#2A78D6", "Node.js": "#1BAF7A", ".NET": "#4A3AA7"}
TECH_FALLBACK = "#90A4AE"

if eol_df.empty:
    fig = qu.empty_figure("No end-of-life dependencies found for the selected filters.")
else:
    df = eol_df.copy()
    df["isEol"] = df["eol"].astype(str).str.strip().str.lower() == "true"

    # One row per distinct component occurrence per repository. Also collapses
    # components reported by both the platform and the language-specific recipes.
    comp = df.drop_duplicates(
        subset=["repoShort", "technology", "componentId", "version"]
    )

    repos_affected = int(comp["repoShort"].nunique())
    already = int(comp["isEol"].sum())
    approaching = int((~comp["isEol"]).sum())
    products = int(comp["product"].nunique())

    # Panel A: repositories ranked by number of distinct EOL components.
    by_repo = comp.groupby(["repoShort", "isEol"]).size().unstack(fill_value=0)
    for col in (True, False):
        if col not in by_repo.columns:
            by_repo[col] = 0
    by_repo["total"] = by_repo[True] + by_repo[False]
    top_repo = by_repo.sort_values("total", ascending=True).tail(15)

    # Panel B: distribution of EOL component occurrences across technologies.
    tech = comp.groupby("technology").size().sort_values(ascending=False)

    # Panel C: EOL products ranked by number of repositories that still use them.
    prod_repos = (
        comp.groupby("product")["repoShort"]
        .nunique()
        .sort_values(ascending=True)
        .tail(12)
    )

    fig = make_subplots(
        rows=3,
        cols=4,
        specs=[
            [
                {"type": "indicator"},
                {"type": "indicator"},
                {"type": "indicator"},
                {"type": "indicator"},
            ],
            [{"type": "bar", "colspan": 3}, None, None, {"type": "domain"}],
            [{"type": "bar", "colspan": 4}, None, None, None],
        ],
        row_heights=[0.20, 0.44, 0.36],
        vertical_spacing=0.16,
        subplot_titles=[
            "",
            "",
            "",
            "",
            "Repositories most exposed to end-of-life components"
            "<br><sup>distinct components per repository — already end-of-life vs approaching</sup>",
            "By technology" "<br><sup>EOL component share</sup>",
            "End-of-life products by number of repositories affected"
            "<br><sup>how many repositories still rely on each end-of-life product</sup>",
        ],
    )

    repos_suffix = (
        f" / {total_repositories_scanned}" if total_repositories_scanned else ""
    )
    title_font = {"size": 13, "color": "#555"}
    # Pin a uniform number font so the digit count never changes the size — otherwise
    # Plotly auto-fits each indicator and shrinks the larger (most important) numbers.
    NUM_SIZE = 46
    fig.add_trace(
        go.Indicator(
            mode="number",
            value=repos_affected,
            number={
                "suffix": repos_suffix,
                "font": {"size": NUM_SIZE, "color": "#37474F"},
            },
            title={"text": "Repos using<br>EOL components", "font": title_font},
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Indicator(
            mode="number",
            value=already,
            number={"font": {"color": RED, "size": NUM_SIZE}},
            title={"text": "Components<br>already EOL", "font": title_font},
        ),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Indicator(
            mode="number",
            value=approaching,
            number={"font": {"color": AMBER, "size": NUM_SIZE}},
            title={"text": "Approaching<br>EOL", "font": title_font},
        ),
        row=1,
        col=3,
    )
    fig.add_trace(
        go.Indicator(
            mode="number",
            value=products,
            number={"font": {"size": NUM_SIZE, "color": "#37474F"}},
            title={"text": "Distinct EOL<br>products", "font": title_font},
        ),
        row=1,
        col=4,
    )

    eol_text = [str(v) if v else "" for v in top_repo[True]]
    soon_text = [str(v) if v else "" for v in top_repo[False]]
    fig.add_trace(
        go.Bar(
            y=list(top_repo.index),
            x=list(top_repo[True]),
            name="Already end-of-life",
            orientation="h",
            marker_color=RED,
            text=eol_text,
            textposition="inside",
            hovertemplate="%{y}: %{x} already-EOL components<extra></extra>",
        ),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Bar(
            y=list(top_repo.index),
            x=list(top_repo[False]),
            name="Approaching end-of-life",
            orientation="h",
            marker_color=AMBER,
            text=soon_text,
            textposition="inside",
            hovertemplate="%{y}: %{x} approaching-EOL components<extra></extra>",
        ),
        row=2,
        col=1,
    )

    fig.add_trace(
        go.Pie(
            labels=list(tech.index),
            values=[int(v) for v in tech.values],
            hole=0.45,
            sort=False,
            direction="clockwise",
            marker={
                "colors": [TECH_COLORS.get(t, TECH_FALLBACK) for t in tech.index],
                "line": {"color": "#FFFFFF", "width": 2},
            },
            textinfo="label+percent",
            textposition="auto",
            insidetextorientation="horizontal",
            showlegend=False,
            hovertemplate="%{label}: %{value} EOL component occurrences (%{percent})<extra></extra>",
        ),
        row=2,
        col=4,
    )
    # Shrink the donut so outside slice labels clear the subplot title above it.
    pie = next(t for t in fig.data if t.type == "pie")
    pie.domain.y = (pie.domain.y[0], pie.domain.y[1] - 0.06)

    fig.add_trace(
        go.Bar(
            y=list(prod_repos.index),
            x=list(prod_repos.values),
            orientation="h",
            marker={
                "color": list(prod_repos.values),
                "colorscale": "Reds",
                "cmin": 0,
                "cmax": float(max(prod_repos.values)),
            },
            text=list(prod_repos.values),
            textposition="outside",
            showlegend=False,
            hovertemplate="%{y}: used by %{x} repositories<extra></extra>",
        ),
        row=3,
        col=1,
    )

    plural = "y" if repos_affected == 1 else "ies"
    share = (
        f" ({repos_affected / total_repositories_scanned:.0%} of the portfolio)"
        if total_repositories_scanned
        else ""
    )
    fig.update_layout(
        barmode="stack",
        height=900,
        template="plotly_white",
        title={
            "text": f"End-of-life exposure — {repos_affected} repositor{plural} rely on end-of-life components{share}",
            "x": 0.5,
            "xanchor": "center",
            "y": 0.975,
            "font": {"size": 17},
        },
        legend={"orientation": "h", "y": -0.07, "x": 0.5, "xanchor": "center"},
        margin={"t": 150, "l": 175, "r": 55, "b": 70},
    )
    # Show every product label even when many products fit in the bottom panel.
    fig.update_yaxes(dtick=1, row=3, col=1)

fig.show()